# 03 · Fine-Tuning com MultipleNegativesRankingLoss

Ajusta o modelo pré-treinado ao domínio do WikiIR 1k.

**Loss:** `MultipleNegativesRankingLoss`
- Usa pares `(query, doc_positivo)`
- Demais exemplos no batch atuam como **negativos implícitos**
- Estado-da-arte para tarefas de retrieval denso

## 1 · Instalação

In [1]:
#!py -m pip install sentence-transformers torch

## 2 · Imports e configuração

In [2]:
import json, math
from pathlib import Path
import torch
from tqdm import tqdm
from sentence_transformers import (
    InputExample, SentenceTransformer,
    losses, evaluation,
)
from torch.utils.data import DataLoader

BASE_MODEL   = 'sentence-transformers/all-MiniLM-L6-v2'
OUTPUT_DIR   = Path('models/finetuned')
DATA_DIR     = Path('data')
EPOCHS       = 30
BATCH_SIZE   = 32
WARMUP_RATIO = 0.1
LR           = 2e-5
EVAL_STEPS   = 200
MAX_SEQ_LEN  = 256

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device disponível: {device}')

Device disponível: cuda


C:\Users\fsbertoglio\AppData\Local\Temp\ipykernel_14512\3071399825.py:5: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import (
C:\Users\fsbertoglio\AppData\Local\Temp\ipykernel_14512\3071399825.py:5: DeprecationWarning: Importing from 'sentence_transformers.evaluation' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.evaluation' instead.
  from sentence_transformers import (


## 3 · Carregamento dos exemplos de treino

In [3]:
def load_examples(path, use_triplets=False):
    examples = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            item = json.loads(line)
            if use_triplets and 'negative' in item:
                examples.append(InputExample(
                    texts=[item['query'], item['positive'], item['negative']]))
            else:
                examples.append(InputExample(
                    texts=[item['query'], item['positive']]))
    return examples

train_examples = load_examples(DATA_DIR / 'train_hard_triplets.jsonl', use_triplets=True)
print(f'Exemplos de treino: {len(train_examples):,}')
print(f'Usando hard negatives: {len(train_examples[0].texts) == 3}')

Exemplos de treino: 3,899
Usando hard negatives: True


## 4 · Modelo base

In [4]:
print(f'Carregando modelo: {BASE_MODEL}')
model = SentenceTransformer(BASE_MODEL, device=device)
model.max_seq_length = MAX_SEQ_LEN
print('Pronto!')

Carregando modelo: sentence-transformers/all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Pronto!


## 5 · DataLoader e Loss

In [5]:
train_loader = DataLoader(
    train_examples,
    shuffle=True,
    batch_size=BATCH_SIZE,
    drop_last=True,
    collate_fn=lambda batch: batch,  # retorna lista de InputExample
)

train_loss = losses.MultipleNegativesRankingLoss(model)
print(f'Loss: MultipleNegativesRankingLoss | {len(train_loader)} steps/epoch')

Loss: MultipleNegativesRankingLoss | 121 steps/epoch


## 6 · Avaliador de IR (InformationRetrievalEvaluator)

In [6]:
def build_ir_evaluator(data_dir):
    queries, corpus, relevant_docs = {}, {}, {}

    with open(data_dir / 'queries.jsonl') as f:
        for line in f:
            item = json.loads(line)
            queries[item['query_id']] = item['text']

    with open(data_dir / 'corpus.jsonl') as f:
        for line in f:
            item = json.loads(line)
            corpus[item['doc_id']] = item['text']

    relevant_docs = {qid: set() for qid in queries}
    with open(data_dir / 'qrels.txt') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 4:
                qid, _, did, rel = parts
                if qid in relevant_docs and int(rel) >= 1:
                    relevant_docs[qid].add(did)

    relevant_docs = {k: v for k, v in relevant_docs.items() if v}

    return evaluation.InformationRetrievalEvaluator(
        queries=queries, corpus=corpus, relevant_docs=relevant_docs,
        name='wikiir-val',
        mrr_at_k=[10], ndcg_at_k=[10], map_at_k=[10, 100],
        show_progress_bar=False,
    )

evaluator = build_ir_evaluator(DATA_DIR)
print('Avaliador criado!')

Avaliador criado!


## 7 · Configuração do scheduler

In [7]:
total_steps  = len(train_loader) * EPOCHS
warmup_steps = math.ceil(total_steps * WARMUP_RATIO)

print(f'Steps totais : {total_steps:,}')
print(f'Warmup steps : {warmup_steps}')
print(f'Epochs       : {EPOCHS}')
print(f'Batch size   : {BATCH_SIZE}')
print(f'Learning rate: {LR}')

Steps totais : 3,630
Warmup steps : 363
Epochs       : 30
Batch size   : 32
Learning rate: 2e-05


## 8 · Fine-Tuning

> ⚠️ **Tempo estimado:** ~30 min (GPU) / 2–4 h (CPU)  
> Para testes rápidos, reduza `EPOCHS = 1`.

In [8]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

optimizer = AdamW(model.parameters(), lr=LR)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)
scaler = torch.amp.GradScaler('cuda', enabled=(device == 'cuda'))

def primary_score(metrics):
    key = [k for k in metrics if 'ndcg' in k.lower()][0]
    return metrics[key]

use_hard_negs = len(train_examples[0].texts) == 3
print(f'Hard negatives: {use_hard_negs}')

best_score = -1
global_step = 0

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0

    for batch in tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS}'):
        queries   = [ex.texts[0] for ex in batch]
        positives = [ex.texts[1] for ex in batch]

        feat_q = {k: v.to(device) for k, v in model.tokenize(queries).items() if isinstance(v, torch.Tensor)}
        feat_p = {k: v.to(device) for k, v in model.tokenize(positives).items() if isinstance(v, torch.Tensor)}
        features = [feat_q, feat_p]

        if use_hard_negs:
            hard_negs = [ex.texts[2] for ex in batch]
            feat_n = {k: v.to(device) for k, v in model.tokenize(hard_negs).items() if isinstance(v, torch.Tensor)}
            features.append(feat_n)

        optimizer.zero_grad()
        with torch.autocast(device_type=device, enabled=(device == 'cuda')):
            loss_val = train_loss(features, None)

        scaler.scale(loss_val).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        epoch_loss += loss_val.item()
        global_step += 1

        if global_step % EVAL_STEPS == 0:
            metrics = evaluator(model, output_path=str(OUTPUT_DIR))
            score = primary_score(metrics)
            print(f'Step {global_step} | nDCG@10={score:.4f} | best={best_score:.4f}')
            if score > best_score:
                best_score = score
                model.save(str(OUTPUT_DIR))
                print('  → Melhor modelo salvo!')
            model.train()

    avg_loss = epoch_loss / len(train_loader)
    print(f'Epoch {epoch+1}/{EPOCHS} | loss={avg_loss:.4f}')

metrics = evaluator(model, output_path=str(OUTPUT_DIR))
score = primary_score(metrics)
print(f'\nMétricas finais:')
for k, v in metrics.items():
    print(f'  {k}: {v:.4f}')
if score > best_score:
    model.save(str(OUTPUT_DIR))

print(f'\nFine-tuning concluído! Modelo salvo em {OUTPUT_DIR}')

Hard negatives: True


Epoch 1/30:   0%|          | 0/121 [00:00<?, ?it/s]The `tokenize` method is deprecated, please use `preprocess` instead.
c:\Users\fsbertoglio\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\optim\lr_scheduler.py:227: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(
Epoch 1/30: 100%|██████████| 121/121 [00:12<00:00,  9.88it/s]


Epoch 1/30 | loss=3.3183


Epoch 2/30:  64%|██████▍   | 78/121 [00:17<00:04, 10.68it/s]

Step 200 | nDCG@10=0.1325 | best=-1.0000


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 30/30:  75%|███████▌  | 91/121 [04:44<24:26, 48.88s/it]

  → Melhor modelo salvo!


Epoch 30/30: 100%|██████████| 121/121 [04:47<00:00,  2.38s/it]


Epoch 30/30 | loss=0.1110

Métricas finais:
  wikiir-val_cosine_accuracy@1: 0.4307
  wikiir-val_cosine_accuracy@3: 0.6683
  wikiir-val_cosine_accuracy@5: 0.7659
  wikiir-val_cosine_accuracy@10: 0.8566
  wikiir-val_cosine_precision@1: 0.4307
  wikiir-val_cosine_precision@3: 0.3740
  wikiir-val_cosine_precision@5: 0.3248
  wikiir-val_cosine_precision@10: 0.2463
  wikiir-val_cosine_recall@1: 0.0449
  wikiir-val_cosine_recall@3: 0.1168
  wikiir-val_cosine_recall@5: 0.1642
  wikiir-val_cosine_recall@10: 0.2353
  wikiir-val_cosine_ndcg@10: 0.3221
  wikiir-val_cosine_mrr@10: 0.5708
  wikiir-val_cosine_map@10: 0.1963
  wikiir-val_cosine_map@100: 0.2003

Fine-tuning concluído! Modelo salvo em models\finetuned


## 9 · Verificação do modelo salvo

In [9]:
from sentence_transformers import SentenceTransformer as ST

ft_model = ST(str(OUTPUT_DIR))
test_emb = ft_model.encode(['teste de embedding'])
print(f'Modelo fine-tuned carregado com sucesso!')
print(f'Dimensão do embedding: {test_emb.shape[1]}')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Modelo fine-tuned carregado com sucesso!
Dimensão do embedding: 384
